In [1]:
import os
import torch
from torch import nn
import torch.optim as optim

import torch.utils.data
from torch.utils.data import SubsetRandomSampler, DataLoader
from sklearn.model_selection import KFold
from tqdm import tqdm

import pandas as pd
import numpy as np

import joblib
from joblib import Parallel, delayed
import os

In [2]:
class Module1Layer(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.dense0 = nn.Linear(n_features, n_neurons)
        self.act = nonlin
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.sigmoid(self.output(X))
        return X

class Module2Layers(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.dense0 = nn.Linear(n_features, n_neurons)
        self.act = nonlin
        self.dense1 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.act(self.dense1(X))
        X = self.sigmoid(self.output(X))
        return X

class Module3Layers(nn.Module):
    def __init__(self, n_features=8, n_neurons=16, nonlin=nn.ReLU()):
        super().__init__()

        self.act = nonlin
        self.dense0 = nn.Linear(n_features, n_neurons)
        self.dense1 = nn.Linear(n_neurons, n_neurons)
        self.dense2 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X, **kwargs):
        X = self.act(self.dense0(X))
        X = self.act(self.dense1(X))
        X = self.act(self.dense2(X))
        X = self.sigmoid(self.output(X))
        return X

In [3]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        # create feature tensors
        self.features = torch.tensor(X, dtype=torch.float32)
        # create label tensors
        self.labels = torch.tensor(y, dtype=torch.long) 

    def __len__(self):
        # we define a method to retrieve the length of the dataset
        return self.features.shape[0]

    def __getitem__(self, idx):
        # necessary override of the __getitem__ method which helps to index our data
        x = self.features[idx]
        y = self.labels[idx]
        return x, y

In [4]:
def run_training(dataset_name):

    seed = 42
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
    num_epochs = 128
    batch_size = 16 
    lr = 0.01
    n_neurons = 256
    fun_nonlim = nn.ReLU()
    model_name = 'Module3Layers' #todo
    
    X = pd.read_csv('./data/' + dataset_name +'/processed/X_train.csv', delimiter=',')
    y = pd.read_csv('./data/' + dataset_name +'/processed/y_train.csv', delimiter=',')
    simple_dataset = Dataset(X.values, y.values)
        
        
    # Configuration options
    k_folds = 5
    kfold = KFold(n_splits=k_folds, shuffle=True)
        
    loss_list = []
    acc_list = []
        
    for fold, (train_ids, valid_ids) in enumerate(kfold.split(simple_dataset)):
        # print(f'FOLD {fold}')
        # print('--------------------------------')
        train_subsampler = SubsetRandomSampler(train_ids)
        valid_subsampler = SubsetRandomSampler(valid_ids)
            
        train_loader = DataLoader(simple_dataset, batch_size=batch_size, sampler=train_subsampler)
        valid_loader = DataLoader(simple_dataset, batch_size=batch_size, sampler=valid_subsampler)
            
        simple_nn = Module3Layers(n_features=X.shape[1], n_neurons=n_neurons, nonlin=fun_nonlim) #todo
        optimizer = optim.Adamax(simple_nn.parameters(), lr=lr)
        error = nn.BCELoss()

        for epoch in range(num_epochs):
            with torch.no_grad():
                valid_loss = 0
                num_right = 0
                for tensor_x, tensor_y in valid_loader:
                    tensor_x = tensor_x.float()
                    tensor_y = tensor_y.float().reshape(-1, 1)
                    output = simple_nn(tensor_x)
                    loss = error(output, tensor_y)
                    valid_loss += loss.item() * len(tensor_x)
                    result = [1 if out >= 0.5 else 0 for out in output]
                    num_right += np.sum(np.array(result) == tensor_y.numpy().reshape(-1))
                    
                valid_loss = valid_loss / len(valid_loader.sampler.indices)
                valid_accuracy = num_right / len(valid_loader.sampler.indices)
                
        
            train_loss = 0
            num_right = 0
            for tensor_x, tensor_y in train_loader:
                tensor_x = tensor_x.float()
                tensor_y = tensor_y.float().reshape(-1, 1)
                optimizer.zero_grad()
                output = simple_nn(tensor_x)
                loss = error(output, tensor_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * len(tensor_x)
                result = [1 if out >= 0.5 else 0 for out in output]
                num_right += np.sum(np.array(result) == tensor_y.numpy().reshape(-1))
                    
            train_loss = train_loss / len(train_loader.sampler.indices)
            accuracy = num_right / len(train_loader.sampler.indices)

                    
        loss_list.append(valid_loss)
        acc_list.append(valid_accuracy)


    torch.save(simple_nn.state_dict(), './saved_models/' + dataset_name  +'/' + model_name + '.pt')